# 🛡️ Federated Learning Multi-Scenario Benchmark Runner (FL-IoT-IDS)
## End-to-End Comparative Evaluation: Scenario E1 vs E2 vs E5
### Master's Thesis: *Xây dựng và đánh giá prototype hệ thống phát hiện xâm nhập IoT sử dụng Federated Learning trong môi trường dữ liệu non-IID*
**Author:** Nguyễn Việt Kỳ Quân (2026) | **Institution:** HCMUT - VNU-HCM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NVKQ2022/FedLearning/blob/main/notebooks/colab_experiment_runner.ipynb)

---

### 🎯 Purpose in Thesis
This notebook provides an automated, one-click master benchmark runner that trains and compares the three seminal experimental scenarios of this thesis back-to-back under strictly identical holdout test conditions:

1. **Scenario E1 (Centralized Baseline):** Upper bound detection ceiling without decentralization.
2. **Scenario E2 (Federated IID FedAvg):** Decentralized edge benchmark with uniform data distribution.
3. **Scenario E5 (Federated Non-IID FedProx):** Extreme statistical heterogeneity (Dirichlet $\alpha = 0.1$) stabilized by proximal regularization $\mu = 0.05$.

Generates a consolidated comparative scoreboard measuring the exact impact of non-IID label skew on detection accuracy, **Macro-F1**, and stealthy intrusion **Minority Recall**.

---
### 🛠️ Step 1: Environment Setup & Hardware Acceleration
We verify GPU acceleration, synchronize repository code in Colab, and enforce deterministic seeds.

In [ ]:
import os, sys, subprocess, copy, warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Compute Device: {device} | PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'   Active GPU: {torch.cuda.get_device_name(0)}')

# Colab setup
if os.path.exists('/content'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        pass
    repo_dir = '/content/FedLearning'
    if not os.path.exists(repo_dir):
        subprocess.run(['git', 'clone', 'https://github.com/NVKQ2022/FedLearning.git', repo_dir], check=True)
    os.chdir(repo_dir)
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
elif os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from src.utils.seed import set_seed
set_seed(42)
print('✅ Deterministic random seed locked at 42.')

---
### ⚙️ Step 2: Global Configuration Container
Unified hyperparameter container controlling all 3 benchmark scenarios.

In [ ]:
from src.utils.config import (
    Experiment,
    DataConfig,
    ModelConfig,
    LossConfig,
    OptimizerConfig,
    FederatedConfig,
)

CONFIG = Experiment(
    experiment_name="colab_multi_scenario_benchmark",
    seed=42,
    data=DataConfig(
        sample_size=60000,                  # 60k for rapid all-scenario execution
        test_size=0.2,                      # 20% holdout test set (unseen by all scenarios)
        val_size=0.1,                       # 10% validation set
        scaler_type="robust",               # 'robust' (median/IQR)
        batch_size=128,
    ),
    model=ModelConfig(
        hidden_dims=(128, 64),              # Lightweight edge MLP (~15k params)
        dropout_rate=0.2,
    ),
    loss=LossConfig(
        loss_type="focal_loss",             # MultiClassFocalLoss
        class_weight_strategy="balanced",   # Inverse frequency weighting
        focal_gamma=2.0,
    ),
    optimizer=OptimizerConfig(
        epochs=8,
        patience=3,
        learning_rate=1e-3,
        weight_decay=1e-4,
    ),
    federated=FederatedConfig(
        num_clients=5,                      # Number of IoT edge clients K
        num_rounds=8,                       # Communication rounds T
        local_epochs=2,                     # Local epochs E
        mu=0.05,                            # FedProx proximal coefficient (Scenario E5)
        dirichlet_alpha=0.1,                # Severe Non-IID Dirichlet concentration (Scenario E5)
    ),
)

CONFIG.display()


---
### 💾 Step 3: Leak-Free Preprocessing & Shared Holdout Test Set
We preprocess the dataset once. All three scenarios are benchmarked against the identical stationary holdout test set.

In [ ]:
from src.data.preprocess import load_and_preprocess_ciciot2023, compute_balanced_class_weights

candidate_paths = [
    CONFIG.gdrive_csv,
    CONFIG.alt_gdrive_csv,
    CONFIG.target_csv_path,
    'datasets/CICIOT2023/merged_CICIOT2023_data.csv',
    'datasets/merged_CICIOT2023_data.csv',
]
csv_path = next((p for p in candidate_paths if os.path.exists(p) and os.path.getsize(p) > 10*1024*1024), None)
if not csv_path:
    csv_path = CONFIG.target_csv_path

print(f"📁 Loading dataset from: {csv_path}")
data = load_and_preprocess_ciciot2023(
    csv_path=csv_path,
    test_size=CONFIG.test_size,
    val_size=CONFIG.val_size,
    sample_size=CONFIG.sample_size,
    scaler_type=CONFIG.scaler_type,
    batch_size=CONFIG.batch_size,
    random_state=CONFIG.seed
)

class_weights = compute_balanced_class_weights(data['y_train'], data['num_classes'], strategy=CONFIG.class_weight_strategy)
class_names = data['class_names']
minority_classes = ['Web-based', 'Brute-force']

print(f"✅ Training samples: {len(data['X_train']):,} | Test samples: {len(data['X_test']):,}")
print(f"✅ Classes: {class_names}")

---
### 🏆 Step 4: Scenario E1 - Centralized Upper Bound Baseline
Train `TabularIoTMLP` on pooled data to establish the empirical upper bound ceiling.

In [ ]:
from src.models.mlp import TabularIoTMLP
from src.losses.focal_loss import build_loss_function
from src.optimizers.optimizer import build_optimizer
from src.training.trainer import CentralizedTrainer
from src.evaluation.evaluator import evaluate_comprehensive

print("Starting Scenario E1: Centralized Training...")
model_e1 = TabularIoTMLP(data['input_dim'], CONFIG.hidden_dims, data['num_classes'], CONFIG.dropout_rate).to(device)
criterion = build_loss_function(CONFIG.loss_type, class_weights=class_weights, gamma=CONFIG.focal_gamma, device=device)
opt_e1 = build_optimizer(model_e1, CONFIG.optimizer_type, lr=CONFIG.learning_rate, weight_decay=CONFIG.weight_decay)

trainer_e1 = CentralizedTrainer(model_e1, opt_e1, criterion, device=device, max_grad_norm=CONFIG.max_grad_norm)
trainer_e1.fit(data['train_loader'], data['val_loader'], epochs=CONFIG.epochs, patience=CONFIG.patience, verbose=False)

res_e1 = evaluate_comprehensive(model_e1, data['test_loader'], class_names, minority_classes=minority_classes, device=device)
print(f"✅ Scenario E1 Centralized Test Macro-F1: {res_e1['macro_f1']*100:.2f}%")
print(f"   Minority Recall: {res_e1['minority_recall']}")

---
### 🌐 Step 5: Scenario E2 - Federated IID Baseline (FedAvg)
Simulate uniform stratified data distribution across $K=5$ clients with standard FedAvg.

In [ ]:
from src.data.partitioner import StratifiedIIDPartitioner, create_client_dataloaders
from src.training.federated_trainer import FederatedClientTrainer
from src.federated.strategy import FedAvgStrategy

print("Starting Scenario E2: Federated IID FedAvg...")
part_iid = StratifiedIIDPartitioner(num_clients=CONFIG.num_clients, seed=CONFIG.seed).partition(data['X_train'], data['y_train'])
loaders_iid = create_client_dataloaders(data['X_train'], data['y_train'], part_iid, batch_size=CONFIG.batch_size)

model_e2 = TabularIoTMLP(data['input_dim'], CONFIG.hidden_dims, data['num_classes'], CONFIG.dropout_rate).to(device)
strat_e2 = FedAvgStrategy()

for r in range(1, CONFIG.num_rounds + 1):
    c_res = []
    for cid in range(CONFIG.num_clients):
        loc_m = copy.deepcopy(model_e2)
        loc_opt = build_optimizer(loc_m, CONFIG.optimizer_type, lr=CONFIG.learning_rate, weight_decay=CONFIG.weight_decay)
        t_c = FederatedClientTrainer(loc_m, loc_opt, criterion, device=device)
        loss_c, acc_c = t_c.train_epochs(loaders_iid[cid], num_epochs=CONFIG.local_epochs, mu=0.0)
        c_res.append((loc_m.get_weights(), len(part_iid[cid]), {"loss": loss_c, "acc": acc_c}))
    agg_w, _ = strat_e2.aggregate_fit(r, c_res)
    model_e2.set_weights(agg_w)

res_e2 = evaluate_comprehensive(model_e2, data['test_loader'], class_names, minority_classes=minority_classes, device=device)
print(f"✅ Scenario E2 IID FedAvg Test Macro-F1: {res_e2['macro_f1']*100:.2f}%")
print(f"   Minority Recall: {res_e2['minority_recall']}")

---
### ⚡ Step 6: Scenario E5 - Severe Non-IID Dirichlet Skew (FedProx $\mu=0.05$)
Simulate extreme statistical heterogeneity (Dirichlet $\alpha = 0.1$) where clients have missing classes, stabilized by FedProx proximal regularization.

In [ ]:
from src.data.partitioner import DirichletNonIIDPartitioner
from src.federated.strategy import FedProxStrategy

print(f"Starting Scenario E5: Non-IID Dirichlet (alpha={CONFIG.dirichlet_alpha}, FedProx mu={CONFIG.mu})...")
part_noniid = DirichletNonIIDPartitioner(num_clients=CONFIG.num_clients, alpha=CONFIG.dirichlet_alpha, seed=CONFIG.seed).partition(data['X_train'], data['y_train'])
loaders_noniid = create_client_dataloaders(data['X_train'], data['y_train'], part_noniid, batch_size=CONFIG.batch_size)

model_e5 = TabularIoTMLP(data['input_dim'], CONFIG.hidden_dims, data['num_classes'], CONFIG.dropout_rate).to(device)
strat_e5 = FedProxStrategy(mu=CONFIG.mu)

for r in range(1, CONFIG.num_rounds + 1):
    c_res = []
    for cid in range(CONFIG.num_clients):
        loc_m = copy.deepcopy(model_e5)
        loc_opt = build_optimizer(loc_m, CONFIG.optimizer_type, lr=CONFIG.learning_rate, weight_decay=CONFIG.weight_decay)
        t_c = FederatedClientTrainer(loc_m, loc_opt, criterion, device=device)
        loss_c, acc_c = t_c.train_epochs(loaders_noniid[cid], num_epochs=CONFIG.local_epochs, global_model=model_e5, mu=CONFIG.mu)
        c_res.append((loc_m.get_weights(), len(part_noniid[cid]), {"loss": loss_c, "acc": acc_c}))
    agg_w, _ = strat_e5.aggregate_fit(r, c_res)
    model_e5.set_weights(agg_w)

res_e5 = evaluate_comprehensive(model_e5, data['test_loader'], class_names, minority_classes=minority_classes, device=device)
print(f"✅ Scenario E5 Non-IID FedProx Test Macro-F1: {res_e5['macro_f1']*100:.2f}%")
print(f"   Minority Recall: {res_e5['minority_recall']}")

---
### 📊 Step 7: Master Comparative Benchmark Scoreboard
Consolidated comparison across Scenario E1, E2, and E5.

In [ ]:
# 1. Consolidated Comparison Table
comparison_df = pd.DataFrame([
    {
        "Scenario": "E1: Centralized (Upper Bound)",
        "Architecture / Strategy": "Centralized TabularIoTMLP",
        "Accuracy (%)": f"{res_e1['accuracy']*100:.2f}%",
        "Macro-F1 (%)": f"{res_e1['macro_f1']*100:.2f}%",
        "Web-based Recall (%)": f"{res_e1['minority_recall']['Web-based']*100:.2f}%",
        "Brute-force Recall (%)": f"{res_e1['minority_recall']['Brute-force']*100:.2f}%"
    },
    {
        "Scenario": "E2: Federated IID",
        "Architecture / Strategy": "FedAvg (K=5, T=8)",
        "Accuracy (%)": f"{res_e2['accuracy']*100:.2f}%",
        "Macro-F1 (%)": f"{res_e2['macro_f1']*100:.2f}%",
        "Web-based Recall (%)": f"{res_e2['minority_recall']['Web-based']*100:.2f}%",
        "Brute-force Recall (%)": f"{res_e2['minority_recall']['Brute-force']*100:.2f}%"
    },
    {
        "Scenario": "E5: Severe Non-IID (α=0.1)",
        "Architecture / Strategy": f"FedProx (μ={CONFIG.mu})",
        "Accuracy (%)": f"{res_e5['accuracy']*100:.2f}%",
        "Macro-F1 (%)": f"{res_e5['macro_f1']*100:.2f}%",
        "Web-based Recall (%)": f"{res_e5['minority_recall']['Web-based']*100:.2f}%",
        "Brute-force Recall (%)": f"{res_e5['minority_recall']['Brute-force']*100:.2f}%"
    }
])

print('=' * 85)
print('🏆 MASTER BENCHMARK SCOREBOARD (E1 vs E2 vs E5)')
print('=' * 85)
display(comparison_df)

# 2. Comparative Bar Chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5), dpi=120)
scens = ['E1: Centralized', 'E2: FedAvg (IID)', f'E5: FedProx (α={CONFIG.dirichlet_alpha})']
f1_scores = [res_e1['macro_f1']*100, res_e2['macro_f1']*100, res_e5['macro_f1']*100]
colors = ['#27ae60', '#2980b9', '#e74c3c']

# Macro-F1
bars1 = ax1.bar(scens, f1_scores, color=colors, width=0.55)
ax1.set_title("Test Macro-F1 Comparison", fontsize=11, fontweight='bold')
ax1.set_ylabel("Macro-F1 (%)", fontsize=10)
ax1.set_ylim(0, 100)
for bar in bars1:
    y = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, y + 2, f"{y:.1f}%", ha='center', fontweight='bold', fontsize=9)

# Stealth Minority Recall
x = np.arange(len(scens))
w = 0.35
web_rec = [res_e1['minority_recall']['Web-based']*100, res_e2['minority_recall']['Web-based']*100, res_e5['minority_recall']['Web-based']*100]
bf_rec = [res_e1['minority_recall']['Brute-force']*100, res_e2['minority_recall']['Brute-force']*100, res_e5['minority_recall']['Brute-force']*100]

ax2.bar(x - w/2, web_rec, w, label='Web-based', color='#8e44ad')
ax2.bar(x + w/2, bf_rec, w, label='Brute-force', color='#d35400')
ax2.set_title("Stealth Attack Minority Recall Isolation", fontsize=11, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(scens)
ax2.set_ylabel("Recall (%)", fontsize=10)
ax2.set_ylim(0, 100)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
### 💾 Step 8: Export Consolidated Master Benchmark Summary
Save `reports/master_benchmark_summary.json` for thesis reporting.

In [ ]:
import json

os.makedirs('reports', exist_ok=True)
summary_export = {
    "experiment": "colab_multi_scenario_benchmark",
    "E1_centralized": {k: v for k, v in res_e1.items() if k != 'confusion_matrix'},
    "E2_federated_iid": {k: v for k, v in res_e2.items() if k != 'confusion_matrix'},
    "E5_federated_noniid": {k: v for k, v in res_e5.items() if k != 'confusion_matrix'},
}
summary_file = "reports/master_benchmark_summary.json"
with open(summary_file, 'w') as f:
    json.dump(summary_export, f, indent=2)

print(f"✅ Exported master benchmark summary to: {summary_file}")
print("\n🎉 Master Benchmark Suite execution complete!")